## This notebook shows how to run and examine a single LID experiment object, and how to run multiple experiments at once.

#### See `Bagging_for_LID.experiment_class` for all of its properties.

In [1]:
from Bagging_for_LID.run_files.final_tasks import *

### Running a single expertiment:

Define parameters:

- **dataset_name** (`str`) : Specifies the data distribution to use to either generate or load data. The preset dataset names are available at `Bagging_for_LID.Datasets.dataset_collections`.
- **n** (`int`) : Number of points to be sampled from the `dataset_name` distribution. Loading a dataset will only work if there is data saved with exactly this many sample points from the distribution.
- **lid** (`int`, default =`None`): If possible, this value will be used in the sampling functions to change the ground truth LID of the `dataset_name` distribution. If `None`, the presets from `Bagging_for_LID.Datasets.DatasetGeneration.data_defaults()` will be used.
- **dim** (`int`, default =`None`): If possible, this value will be used in the sampling functions to change the embedding dimension of the `dataset_name` distribution. If `None`, the presets from `Bagging_for_LID.Datasets.DatasetGeneration.data_defaults()` will be used.
- **estimator_name** (`str`): Name of the estimator to be used in the experiment. The thoroughly tested presets are `'mle'` (MLE), `'mada'` (MADA), and `'tle'` (TLE). 
- **bagging_method** (`str`, default =`None`): Name of the bagging method to be used in the experiment for the `estimator_name` baseline. If `None`, no bagging is applied, the baseline is used. The thoroughly tested other presets are `'bag'` (simple bagging), `'bagw'` (bagging with out-of-bag weights), and `'bagwth'` (bagging with out-of-bag weights with neighborhood threshold adjustment).
- **submethod_0** (`str`): Should generally be `'0'`. Used in case of `bagging_method = 'bagw'` or `bagging_method = 'bagwth'` to determine the strategy for handling no-points in the out-of-bag when calculating the error term. Only the `'0'` case has been thoroughly tested, in which case, estimates for the out-of-bag are set to zero when there are no samples.
- **submethod_error** (`str`): Used in case of `bagging_method = 'bagw'` or `bagging_method = 'bagwth'` to calculate the error between in-bag and out-of-bag estimates. The thoroughly tested presets are `'log_diff'` (absolute logarithmic ratio), or `'diff'` (squared error).
- **k** (`int`): The k-Nearest Neighbor parameter for the baseline LID estimator specified by `estimator_name`. Always used, therefore, always has to be specified.
- **sr** (`float` $\in$ (0,1)): The sampling rate (sr) parameter for subsampling in bagging. Used whenever `bagging_method = 'bag'`, `bagging_method = 'bagw'`, or `bagging_method = 'bagwth'`.
- **Nbag** (`int`): The number of bags (B) parameter for bagging. Used whenever `bagging_method = 'bag'`, `bagging_method = 'bagw'`, or `bagging_method = 'bagwth'`.
- **pre_smooth** (`bool`, default =`False`): If `True`, smoothing is applied across in-bag LID estimates using only the points in the respective bags. Used whenever `bagging_method = 'bag'`, `bagging_method = 'bagw'`, or `bagging_method = 'bagwth'`. If `False`, then it has no effect.
- **post_smooth** (`bool`, default =`False`): Always used if `True`, smoothing is applied to the final LID estimates, using every data point, independently of the estimation method. If `False`, then it has no effect.
- **t** (`float` $\in$ [0,$\infty$)):  Used in case of `bagging_method = 'bagw'` or `bagging_method = 'bagwth'` as an exponent for the errors calculated using the formula specified by `submethod_error`. Has no effect otherwise.

In [2]:
params = {'dataset_name': 'M1_Sphere',
          'n': 5000,
          'lid': None,
          'dim': None,
          'estimator_name': 'mle',
          'bagging_method': 'bag',
          'submethod_0': '0',
          'submethod_error': 'log_diff',
          'k': 20,
          'sr': 0.3,
          'Nbag': 10,
          'pre_smooth': True,
          'post_smooth': False,
          't': 1}

Define the experiment using the above `params` as the following.

In [3]:
experiment = LID_experiment(params=params)

Set `directory` where the generated datasets can be saved.

In [4]:
directory = r"C:\pkls"

Generate or load dataset.

In [5]:
data = experiment.generate_data(load=False, directory=directory)

See the generated or loaded dataset as follows.

In [6]:
print(data[0]) #Spatial coordinates of the generated dataset.
print(data[1]) #Ground truth LID of each point in the generated dataset.
print(data[2]) #Embedding dimension of the generated dataset, aka. data[0].shape[1].

[[ 0.33547057  0.02088157  0.564504   ... -0.15234012  0.26825782
   0.1222369 ]
 [-0.37200724 -0.64942355 -0.12986253 ... -0.15286986 -0.05913785
  -0.09038245]
 [ 0.22771083 -0.37551045  0.30161999 ...  0.31253848 -0.04443582
   0.15792741]
 ...
 [-0.34924269  0.01795704  0.49349274 ...  0.35613443  0.27450324
   0.19254003]
 [-0.43608213  0.21002796  0.36156778 ...  0.32617085  0.30296986
   0.12239554]
 [ 0.02543246 -0.37581727  0.09343968 ... -0.34754101  0.14211084
   0.32106331]]
[10 10 10 ... 10 10 10]
11


The generated or loaded dataset object is saved into the experiment object.

In [7]:
print(experiment.data) #Output of experiment.generate_data().

[array([[ 0.33547057,  0.02088157,  0.564504  , ..., -0.15234012,
         0.26825782,  0.1222369 ],
       [-0.37200724, -0.64942355, -0.12986253, ..., -0.15286986,
        -0.05913785, -0.09038245],
       [ 0.22771083, -0.37551045,  0.30161999, ...,  0.31253848,
        -0.04443582,  0.15792741],
       ...,
       [-0.34924269,  0.01795704,  0.49349274, ...,  0.35613443,
         0.27450324,  0.19254003],
       [-0.43608213,  0.21002796,  0.36156778, ...,  0.32617085,
         0.30296986,  0.12239554],
       [ 0.02543246, -0.37581727,  0.09343968, ..., -0.34754101,
         0.14211084,  0.32106331]]), array([10, 10, 10, ..., 10, 10, 10]), 11, array([10, 10, 10, ..., 10, 10, 10])]


Run the LID estimation experiment as follows.

In [8]:
_ , estimators_results_dictionary = experiment.estimate()

See the output LID estimates and Mean Squared Error (MSE), Bias Squared (Bias^2) and Variance (Var) decomposition as follows.

In [9]:
print(estimators_results_dictionary['mle']['M1_Sphere'][0]) #LID estimates (Using the 'mle' estimator with the chosen bagging method on the points of the 'M1_Sphere' dataset).
print(estimators_results_dictionary['mle']['M1_Sphere'][1]) #Mean Squared Error (MSE) of LID estimates.
print(estimators_results_dictionary['mle']['M1_Sphere'][2]) #Bias Squared (Bias^2) of LID estimates. 
print(estimators_results_dictionary['mle']['M1_Sphere'][3]) #Variance (Var) of LID estimates.
print(estimators_results_dictionary['mle']['M1_Sphere'][4]) #Dictionary of MSE, Var, Bias^2 per manifold (indexed by their ground truth LID).
print(estimators_results_dictionary['mle']['M1_Sphere'][5]) #Dictionary of dataset, split based on different manifolds (indexed by their ground truth LID).

9.891727034427275
0.07083323947295596
0.011723035073912551
0.059110204399043416
{10: [9.891727034427275, 0.07083323947295596, 0.011723035073912551, 0.059110204399043416]}
{10: [array([ 9.86211622, 10.27987902, 10.20657213, ...,  9.81876106,
        9.79964113,  9.940753  ]), array([10, 10, 10, ..., 10, 10, 10])]}


The output LID estimates and Mean Squared Error (MSE), Bias Squared (Bias^2) and Variance (Var) decomposition results are saved into the experiment object. Among other things, such as the set parameters.

In [10]:
print(experiment.lid_estimates) #LID estimates.
print(experiment.total_mse) #Mean Squared Error of LID estimates.
print(experiment.total_bias2) #Bias Squared of LID estimates.
print(experiment.total_var) #Variance of LID estimates.
print(experiment.params) #Parameters of experiment.

[ 9.86211622 10.27987902 10.20657213 ...  9.81876106  9.79964113
  9.940753  ]
0.07083323947295596
0.011723035073912551
0.059110204399043416
{'dataset_name': 'M1_Sphere', 'n': 5000, 'lid': None, 'dim': None, 'estimator_name': 'mle', 'bagging_method': 'bag', 'submethod_0': '0', 'submethod_error': 'log_diff', 'k': 20, 'sr': 0.3, 'Nbag': 10, 'pre_smooth': True, 'post_smooth': False, 't': 1}


### To run multiple experiments at once, across all possible combination of the input parameters, use the `Bagging_for_LID.RunningEstimators.Running2.new_result_generator` function.

#### Define parameter lists instead of single values. Internally, `param_dicts` is expanded into single parameter dictionaries, for every different combinations of the list elements.

In [11]:
param_dicts = {'dataset_name': ['M1_Sphere', 'M2_Affine_3to5'],
              'n': 1000,
              'lid': None,
              'dim': None,
              'estimator_name': ['mle', 'tle'],
              'bagging_method': [None, 'bag'],
              'submethod_0': '0',
              'submethod_error': 'log_diff',
              'k': [10, 20],
              'sr': [0.2, 0.4],
              'Nbag': [5, 10],
              'pre_smooth': False,
              'post_smooth': [True, False],
              't': 1}

We set up the variables for computing/loading the results:

- **load** (`bool`): If `true`, we load already performed (downloaded or saved) experiment objects as .pkl files, from the `pkl_directory` folder.
- **load_data** (`bool`): If `true`, we load already generated datasets as .pkl files, from the `pkl_directory` folder. If `false` every experiment object will generate a new dataset. Note that this variable has an effect on the results only if `load = False`.
- **pkl_directory** (`path`): Path to directory where we save and load datasets and completed experiment objects from as .pkl files.
- **multiprocess** (`bool`): If `true`, we allow the computation across the range of experiments to be paralellized using multiple cores.
- **worker_count** (`int`, default =`None`): If `multiprocess = true`, this determines the number of cores used. The default case means using as many cores as possible.
- **save_name** (`str`): Prefix for the name of all saved experiment objects and plots.

In [12]:
#Setup computation
load=True #Load already complete experiment .pkl files.
load_data=True #Load data for running experiments, if load=True, this has no effect on result output.
pkl_directory = r'C:\pkls' #Directory for saving and loading
multiprocess=True #Toggle multiprocessing
worker_count=None #Worker count for multiprocessing.
save_name='example_result' #Save and load .pkl files from here.

Run every experiment at once as follows.

In [13]:
example_result = new_result_generator(param_dicts, multiprocess=multiprocess, load=load, load_data=load_data,
                                      worker_count=worker_count, save_name=save_name, directory=pkl_directory)

<_io.BufferedReader name='C:\\pkls\\example_result'>


The output `example_result` is a list of experiment objects, one for every parameter combination.

In [14]:
print(example_result)

[<Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD701AF90>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD70139D0>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD7012C90>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FF3E10>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FF2B90>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FED450>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FEF310>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FD4F10>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FD6AD0>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FD06D0>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FD2290>, <Bagging_for_LID.experiment_class.LID_experiment object at 0x0000028CD6FD3E50>, <Bagging_for_LID.experiment_class.LID_e

Experiment lists such as `example_result` are the necessary input format for the plotting functions. However, whether or not the ploting function works will depend on the proper definition of the parameter combinations.

In [15]:
from Bagging_for_LID.Plotting.optimize_across_parameter_results import *

In [16]:
results =  result_extraction(example_result, ['k','sr'], metric_keys=None, directory=save_dir, save=False, decomposition_param=['combined'])

In [17]:
results

{'combined': (               ((Nbag, 10), (bagging_method, None), (estimator_name, mle), (post_smooth, False), (pre_smooth, False), (submethod_0, 0), (submethod_error, log_diff), (t, 1))  \
  M1_Sphere                                               {'k': 20}                                                                                                              
  M2_Affine_3to5                                          {'k': 20}                                                                                                              
  
                 ((Nbag, 10), (bagging_method, bag), (estimator_name, mle), (post_smooth, False), (pre_smooth, False), (submethod_0, 0), (submethod_error, log_diff), (t, 1))  \
  M1_Sphere                                     {'k': 20, 'r': 0.2}                                                                                                             
  M2_Affine_3to5                                {'k': 20, 'r': 0.2}                              